# 次时代崭新：日级样本与七日目标

版本：cyrex-daily-samples-1.0。这个 Notebook 使用本地已经保存的 SteamDT 明确悠悠小时数据，整理每日 10:00 的输入与七日参考标签。**离线构建，不采集、不训练、不输出买卖建议。**

普通版＋崭新出厂为本版研究规格；细磨损与少量普通贴纸不单独建特征，明显属性溢价另作假设。来源价格与七日参考收益不等于实际可成交价格或净利润。

先选择 Python 3.11+ 内核，按顺序运行。默认 `RUN_BUILD=False`，运行全部只读取已有信息；需要新版本时再修改开关并单独运行构建单元格。完整规则见 [字段与操作说明](../../docs/data/CYREX_DAILY_SAMPLES.md)。


In [ ]:
from pathlib import Path
import sys, json, csv, itertools, html
from IPython.display import display, Markdown, HTML

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT = next((p for p in candidates if (p / "scripts/build_research_samples.py").is_file()), None)
if PROJECT is None:
    raise RuntimeError("请先在 VS Code 打开 F:/Codex/CS_YouOweMe 文件夹")
sys.dont_write_bytecode = True
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from scripts import build_research_samples as builder

WORKSPACE = "research_20261016_v1"
OUTPUT_ID = None  # 自动生成新编号；已有目录不覆盖。
SPEC_PATH = PROJECT / "scripts/config/cyrex_daily_spec.json"
print("源库：", PROJECT / "data/local/staging/steamdt" / WORKSPACE / "research.sqlite")
print("规则：", SPEC_PATH)
print("输出根目录：", PROJECT / "data/local/staging/research_samples")


## 1. 先核对规则

每天北京时间 10:00；过去 28 个完整 24 小时段，共 672 小时；目标为 168 小时后结束的最后完整小时收盘与未来路径最低价。

数据不插值、不补零。量额仅备查，本版初始预测特征只使用价格。历史可得时间尚未知，本版保留 `training_approved=false`，不作为严格历史时点回测。


In [ ]:
SPEC = json.loads(SPEC_PATH.read_text(encoding="utf-8"))
display(Markdown("```json\n" + json.dumps(SPEC, ensure_ascii=False, indent=2) + "\n```"))


## 2. 生成一版候选样本

首次构建将下方开关改成 `True`，运行本单元格。它只读本地源库，输出新目录，不覆盖旧结果；不需要 Cookie 或授权参数。构建结束后可把开关改回 `False`。

再次采集数据不会自动改写旧样本。需要更新时先完成一次 SteamDT 有限采集，再在这里生成新样本版本。


In [ ]:
RUN_BUILD = False

if RUN_BUILD:
    BUILD_RESULT = builder.build(workspace=WORKSPACE, output_id=OUTPUT_ID, spec_path=SPEC_PATH)
    print("新输出：", BUILD_RESULT["output_dir"])
    print(json.dumps(BUILD_RESULT.get("summary", {}), ensure_ascii=False, indent=2))
else:
    print("构建关闭。查看已有输出可以继续运行下面的单元格。")


## 3. 查看最新报告

关注样本数、输入完整数、已成熟标签、尚未成熟、未来缺口与跨分组隔离数量。`eligible` 仅表示本版结构规则下可用，尚未批准正式训练。


In [ ]:
LATEST = builder.status()
if LATEST.get("status") == "no_build":
    print("还没有样本输出；需要时运行上方构建单元格。")
else:
    OUTPUT_DIR = Path(LATEST["output_dir"])
    print("当前版本：", OUTPUT_DIR)
    print(json.dumps(LATEST.get("summary", {}), ensure_ascii=False, indent=2))
    REPORT_PATH = OUTPUT_DIR / LATEST.get("report", "report.md")
    display(Markdown(REPORT_PATH.read_text(encoding="utf-8")))


## 4. 只读预览几行

完整 CSV 在上方输出目录。`samples.csv` 每天一行，`sequence.csv` 每个样本有 28 行。空白表示缺失，不能当作 0。下面只读取少量行，便于核对，不做模型拟合。


In [ ]:
def preview_csv(path, count=5, preferred=None):
    with path.open("r", encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle)
        fields = reader.fieldnames or []
        columns = [k for k in (preferred or []) if k in fields] or fields[:10]
        rows = list(itertools.islice(reader, count))
    table = "<table><thead><tr>" + "".join("<th>" + html.escape(k) + "</th>" for k in columns) + "</tr></thead><tbody>"
    table += "".join("<tr>" + "".join("<td>" + html.escape(str(row.get(k, ""))) + "</td>" for k in columns) + "</tr>" for row in rows)
    display(HTML(table + "</tbody></table>"))
    print(path, "；仅预览", len(rows), "行，完整列见 CSV 表头。")

if LATEST.get("status") != "no_build":
    preview_csv(OUTPUT_DIR / "samples.csv", preferred=["decision_at_beijing", "split", "feature_complete", "label_status", "purged", "eligible"])
    preview_csv(OUTPUT_DIR / "sequence.csv", count=3)
else:
    print("尚无可预览样本。")
